# Hydrology pipeline

Real-world DEM-to-streams workflow covering W-1 to W-5, W-10, and W-20:

```text
DEM
  → full_hydro_pipeline   (W-20: fill → flow_direction → accumulate → streams)
  → order(method="hack")           (W-1)
  → order(method="topological")    (W-2)
  → to_vector(...) carrying sinuosity  (W-3)
  → main_stem(...)                 (W-4)
  → prune_short(...)               (W-5)
  → hand(method="euclidean")       (W-10)
```

We build a synthetic mountain catchment with a confluence and exercise every stage on it.

In [ ]:
import numpy as np
from pyramids.dataset import Dataset
from digitalrivers import DEM

rows, cols = 20, 20
z = np.full((rows, cols), 100.0, dtype=np.float32)
for r in range(rows):
    z[r, 10] = float(40 - 2 * r)  # main stem descends south down col 10
for c in range(10):
    z[8, c] = float(35 - c)        # west tributary into (8, 10)
for c in range(11, cols):
    z[13, c] = float(25 - (c - 11))  # east tributary into (13, 10)

ds = Dataset.create_from_array(
    z, top_left_corner=(0.0, 0.0), cell_size=30.0, epsg=32618, no_data_value=-9999.0,
)
dem = DEM(ds.raster)
print(f"DEM shape={z.shape}  elevation range=[{z.min():.1f}, {z.max():.1f}] m")

## W-20 — composite hydro pipeline

`DEM.full_hydro_pipeline(...)` chains fill → flow_direction → accumulate → streams in one call and
returns a dict of typed results.

In [ ]:
bundle = dem.full_hydro_pipeline(
    fill_method="priority_flood",
    flow_method="d8",
    stream_threshold_cells=4,
)
filled = bundle["filled_dem"]
fdir = bundle["flow_direction"]
acc = bundle["accumulation"]
streams = bundle["streams"]

n_stream = int(streams.read_array().astype(bool).sum())
print(f"filled_dem  type={type(filled).__name__}")
print(f"fdir        routing={fdir.routing}")
print(f"acc         max={int(acc.read_array().max())}  routing={acc.routing}")
print(f"streams     cells={n_stream}  threshold={streams.threshold}")

## W-1 — Hack stream ordering

Order 1 is assigned to the main stem (longest source-to-outlet path). Every tributary joining the
main stem becomes order 2; tributaries of those order 3; and so on.

In [ ]:
hack = streams.order(method="hack", flow_direction=fdir)
hack_arr = hack.read_array()
sm = streams.read_array().astype(bool)
print(f"Hack orders on stream cells: {sorted(set(int(v) for v in hack_arr[sm]))}")
print(f"Order-1 (main stem) cells: {int((hack_arr == 1).sum())}")
print(f"Order-2 (1st tributary) cells: {int((hack_arr == 2).sum())}")

## W-2 — Topological stream ordering

Kahn-sort sequential numbering. Heads are enumerated in row-major order first, then progressively
downstream. The outlet's index equals the total stream-cell count.

In [ ]:
topo = streams.order(method="topological", flow_direction=fdir)
topo_arr = topo.read_array()
print(f"Topological max (outlet index): {int(topo_arr.max())}")
print(f"Stream cell count:              {n_stream}")
assert int(topo_arr.max()) == n_stream

## W-3 — Per-link sinuosity

`StreamRaster.to_vector(...)` now carries a `sinuosity` column (traced length / straight-line
distance per link). Straight links have sinuosity exactly 1.0; meandering links exceed 1.0.

In [ ]:
links = streams.to_vector(fdir, dem=filled)
print(links[["link_id", "length_m", "drop_m", "mean_slope", "sinuosity"]])

## W-4 — Main stem

Public API for the longest source-to-outlet path. Returns a binary mask.

In [ ]:
main = streams.main_stem(fdir)
print(f"Main-stem cells: {int(main.sum())} of {n_stream} stream cells")
# Verify Hack-order 1 == main_stem mask
assert (hack_arr[main] == 1).all()

## W-5 — Remove short stream links

`prune_short` drops headwater links below `min_length_m`. Internal links (between two confluences)
stay so the network topology is preserved.

In [ ]:
pruned = streams.prune_short(fdir, min_length_m=120.0)
n_after = int(pruned.read_array().astype(bool).sum())
print(f"Stream cells before prune: {n_stream}")
print(f"Stream cells after  prune: {n_after}")

## W-10 — Euclidean HAND

Height Above Nearest Drainage using 2D Euclidean distance to the nearest stream cell. Stream cells
are 0; cells far from streams have larger HAND.

In [ ]:
hand_eu = filled.hand(streams, method="euclidean")
hand_arr = hand_eu.read_array()
print(f"HAND range on valid cells: [{hand_arr[hand_arr != -9999.0].min():.2f}, "
      f"{hand_arr[hand_arr != -9999.0].max():.2f}] m")
# Stream cells must report HAND = 0
assert (hand_arr[sm] == 0).all()

## Summary

From a single 20×20 synthetic DEM we built:
* A complete hydro pipeline via one `full_hydro_pipeline` call (W-20).
* Two stream-ordering rasters — Hack and Topological (W-1, W-2).
* A vector network with sinuosity per link (W-3).
* A main-stem binary mask (W-4).
* A pruned stream network with short headwater links removed (W-5).
* A Euclidean HAND raster (W-10).